In [2]:
import nest_asyncio
import pandas as pd
from playwright.async_api import async_playwright

In [ ]:
nest_asyncio.apply()
async def scrape_nordpool_lv():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        await page.goto(
            "https://data.nordpoolgroup.com/auction/day-ahead/prices"
            "?deliveryDate=latest"
            "&currency=EUR"
            "&aggregation=DeliveryPeriod"
            "&deliveryAreas=LV",
            wait_until="networkidle"
        )

        await page.wait_for_selector("tr.dx-row", timeout=15000)

        rows = await page.query_selector_all("tr.dx-row")

        data = []
        for r in rows:
            cells = await r.query_selector_all("td")
            if len(cells) != 2:
                continue

            period = (await cells[0].inner_text()).strip()
            price_text = (await cells[1].inner_text()).strip()

            price_text = price_text.replace(",", ".")
            try:
                price = float(price_text)
            except ValueError:
                continue

            data.append({
                "delivery_period": period,
                "price_EUR_MWh": price
            })

        await browser.close()
        return pd.DataFrame(data)

df_np_day_ahead = await scrape_nordpool_lv()

In [14]:
df_np_day_ahead.head()

,delivery_period,price_EUR_MWh
0,01:00 - 01:15,96.81
1,01:15 - 01:30,96.65
2,01:30 - 01:45,94.12
3,01:45 - 02:00,92.65
4,02:00 - 02:15,99.03
